### Import

In [15]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 4

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _, CRATE, DRATE = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)
INEFF_BATT = 0.95
INEFF_INT = 0.99
INEFF_EXT = 0.98

✅ 총 10개 파일을 불러왔습니다: 1033.csv, 1818.csv, 2502.csv, 2503.csv, 2634.csv, 2698.csv, 2816.csv, 545.csv, 665.csv, 690.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=4, M1=3735.41, M2=10638.29
   - 개별 K 값: [200. 200. 400. 600. 100. 600. 300. 200. 300. 200.]


### Individual Optimization (original)

In [16]:
m1 = gp.Model("individual")
m1.setParam("MIPGap", 1e-5)

x_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = m1.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m1.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
m1.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m1.addConstr(R[i, t, s] - (1/INEFF_EXT) * x_ind[i, t] == (1/INEFF_EXT) * yp_ind[i, t, s] - INEFF_EXT * ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= z_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= DRATE[i])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= K[i] - z_ind[i, t, s])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= CRATE[i])
    m1.addConstr(z_ind[i, t, s] <= K[i])
    m1.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + INEFF_BATT * zc_ind[i, t, s] - zd_ind[i, t, s] / INEFF_BATT)

for i, s in product(range(I), range(S)): m1.addConstr(z_ind[i, 0, s] == K0[i])

m1.optimize()

if m1.status == GRB.OPTIMAL:
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; OBJ_IND = m1.objVal
    # phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

Set parameter MIPGap to value 1e-05
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05

Optimize a model with 169000 rows, 121240 columns and 385000 nonzeros
Model fingerprint: 0x1cab907b
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 98923 rows and 27876 columns
Presolve time: 0.13s
Presolved: 70077 rows, 93364 columns, 322308 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.55s

Barrier statistics:
 AA' NZ     : 1.428e+06
 Factor NZ  : 7.639e+06 (roughly 130 MB of memory)
 Factor Ops : 2.371e+09 (less than 1 second per iteration)
 Threads    : 22

         

In [17]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=1
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT) * x_sum:>8.2f} {(1/INEFF_EXT) * yp_avg:>8.2f} {INEFF_EXT * ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |    75.98     0.64    32.53     0.00    42.96     0.15    24.77
 8 |   129.50   100.81    15.82     1.15    26.08    12.07    65.42
 9 |   157.00   113.38    32.76     2.57    27.31    13.89    77.49
10 |   189.17   154.85    29.15     8.97    27.94    13.81    88.82
11 |   226.01   169.09    53.27     8.42    27.78    15.70   100.84
12 |   228.29   175.55    51.78     8.68    24.41    14.77   110.70
13 |   515.87     0.00   556.47     0.00     1.71    42.31   118.34
14 |   467.18     0.00   464.30     0.00    20.41    17.52    75.42
15 |   474.47   461.71    67.55    60.26    26.30    20.83    76.37
16 |   160.22   147.16    15.84     8.68    23.80    17.91    79.43
17 |   191.61   181.32    18.78    16.25    24.52    16.75    83.19
18 |   245.26   195.97    63.31    13.88    18.34    18.48    88.84
19 |   113.65    98.34    19.86

### Holistic Optimization (Linear Decision Rule + MILP - M) (original)

In [18]:
m2 = gp.Model("holistic_MILP_M")
m2.setParam("MIPGap", 1e-5)
m2.setParam(GRB.Param.TimeLimit, 1200)

x_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = m2.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z") ; zc_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m2.update()

obj_lin = gp.quicksum(
    P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)
) + gp.quicksum(
    (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
    for i in range(I) for t in range(T) for s in range(S)
)

eps = 1e-8
# quad_reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
obj = obj_lin - eps * quad_reg

# NOTE
# m2.setObjective(obj, GRB.MAXIMIZE)
m2.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m2.addConstr(R[i, t, s] - (1/INEFF_EXT) * x_hol[i, t] == (1 / INEFF_EXT) * yp_hol[i, t, s] - INEFF_EXT * ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= z_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= DRATE[i])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= K[i] - z_hol[i, t, s])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= CRATE[i])
    m2.addConstr(z_hol[i, t, s] <= K[i])
    m2.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + INEFF_BATT * zc_hol[i, t, s] - zd_hol[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m2.addConstr(z_hol[i, 0, s] == K0[i])

# NOTE : inefficiency 추가
balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = m2.addConstr(INEFF_INT * gp.quicksum(dp_hol[i, t, s] for i in range(I)) == (1/INEFF_INT) * gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

m2.optimize()

if m2.status == GRB.OPTIMAL or m2.status == GRB.TIME_LIMIT:
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = m2.objVal
    OBJ_HOL = sum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + sum(
        (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    QUAD_HOL = eps * sum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

    lambda_dual = np.zeros((T, S))
    for t, s in product(range(T), range(S)): 
        lambda_dual[t, s] = balance_constraints[t, s].Pi
    print("Direct dual extraction successful!")

else:
    print(f"⚠️ Model finished with status: {m2.status}")

Set parameter MIPGap to value 1e-05
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  1200
MIPGap  1e-05

Optimize a model with 171400 rows, 169240 columns and 481000 nonzeros
Model fingerprint: 0xb2319722
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22020 columns
Presolve time: 0.17s
Presolved: 75400 rows, 147220 columns, 430000 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 1.70s
Ordering time: 1.74s

Barrier statistics:
 Dense cols : 220
 AA' NZ     : 4.450e+05
 Factor NZ  : 1.968e+06 (roughly 100 MB of 

In [19]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print(f"\n[HOLISTIC] Objective Value = {OBJ_HOL:.2f}, QUAD_HOL = {QUAD_HOL:>2f}") ; print(header)
for t in range(0, 24):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT) * x_sum:>8.2f} {(1 / INEFF_EXT) * yp_avg:>8.2f} {INEFF_EXT * ym_avg:>8.2f} {INEFF_INT * dp_avg:>8.2f} {(1/INEFF_INT) * dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[HOLISTIC] Objective Value = 5468142.02, QUAD_HOL = 0.034183
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 2 |     0.00     0.00     0.00     0.17     0.00     0.00     0.17     0.00     0.00
 3 |     0.00     0.00     0.00    42.62     0.00     0.00    42.62     0.00     0.16
 4 |     0.00     0.00     0.00   114.33     0.00     0.00   114.33     0.00    40.65
 5 |     0.00     0.00     0.00     5.04     0.00     0.00     5.04     0.00   149.26
 6 |    54.55     0.00     4.25     0.00     6.65     6.65    50.17     0.00   154.06
 7 |   586.20     0.00    76.11     0.00   130.27   130.27   508.13     0.65   201.72
 8 |  1403.93   762.29    29.18     0.00   204.36   204.36   608.34     0

In [20]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_hol[i, t, s] > 0.001 and dm_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, dm={dm_hol[i, t, s]}")
    if zc_hol[i, t, s] > 0.001 and zd_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_hol[i, t, s]}, zd={zd_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.001 and ym_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, ym={ym_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_hol[i, t, s] > 0.001 and yp_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, yp={yp_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

### Individual Replay

In [21]:
data = []

for t in range(T):
    s_fixed = 0
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT_avg': round(P_RT[t, s_fixed], 5), 
        'Lambda': round(-lambda_dual[t, s_fixed] * S, 5), 
        'P_PN_avg': round(P_PN[t, s_fixed], 4)
    })

pd.DataFrame(data)

# data_for_csv = []

# for t in range(T):
#     for s in range(S):
#         row = {
#             "t": t,
#             "s": s,
#             "P_DA": P_DA[t],
#             "P_RT": P_RT[t, s],
#             "P_PN": P_PN[t, s],
#             "P_IN": -lambda_dual[t, s] * S,
#         }
#         data_for_csv.append(row)

# df = pd.DataFrame(data_for_csv)

# output_filename = f"optimization_results_{SEED}.csv"
# df.to_csv(output_filename, index=False, encoding="utf-8-sig")

# print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

,Time,P_DA,P_RT_avg,Lambda,P_PN_avg
0,0,91.140,84.127,184.146,182.286
1,1,80.280,35.884,162.188,160.550
2,2,74.560,32.992,144.748,149.110
3,3,71.640,61.969,144.748,143.286
4,4,71.310,37.963,141.868,142.610
5,5,74.540,76.746,141.868,153.492
6,6,80.820,49.292,81.862,161.642
7,7,87.240,37.423,81.862,174.486
8,8,101.970,44.550,44.100,203.944
9,9,110.370,43.271,42.833,220.740


In [22]:
m5 = gp.Model("DER_Individual_Replay")
# m5.setParam("MIPGap", 1e-5)
m5.setParam(GRB.Param.PoolSearchMode, 1)
m5.setParam(GRB.Param.PoolSolutions, 1)

x = m5.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z = m5.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m5.update()

# NOTE: inefficiency 고려해서 바꿔야함.

obj_lin = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(
        lambda_dual[t, s] * ((1/INEFF_INT) * dm[i, t, s] - INEFF_INT * dp[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
)

eps = 1e-8
# quad_reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s] + dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

obj = obj_lin - eps * quad_reg

# NOTE
# m5.setObjective(obj, GRB.MAXIMIZE)
m5.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m5.addConstr(R[i, t, s] - (1/INEFF_EXT) * x[i, t] == (1 / INEFF_EXT) * yp[i, t, s] - INEFF_EXT * ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= z[i, t, s]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= K[i] - z[i, t, s]) ; m5.addConstr(z[i, t, s] <= K[i])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= DRATE[i]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= CRATE[i])
    m5.addConstr(z[i, t + 1, s] == z[i, t, s] + INEFF_BATT * zc[i, t, s] - zd[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m5.addConstr(z[i, 0, s] == K0[i])

m5.optimize()

if m5.status == GRB.OPTIMAL:
    num_solutions = m5.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = m5.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            m5.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = m5.PoolObjVal
            diff = best_obj - pool_obj

            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    m5.setParam(GRB.Param.SolutionNumber, 0)

    print(f"Optimal solution found! Objective value: {m5.objVal}")
else:
    print("No optimal solution found.")

x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
OBJ_RE = (
    sum(P_DA[t] * x_re[i, t] for i in range(I) for t in range(T))
    + sum(
        (1 / S) * (P_RT[t, s] * yp_re[i, t, s] - P_PN[t, s] * ym_re[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    + sum(
        lambda_dual[t, s]
        * (
            (1/INEFF_INT) * sum(dm_re[i, t, s] for i in range(I))
            - INEFF_INT * sum(dp_re[i, t, s] for i in range(I))
        )
        for t in range(T)
        for s in range(S)
    )
)
QUAD_RE = eps * sum((1 / S) * (dp_re[i, t, s] * dp_re[i, t, s] + dm_re[i, t, s] * dm_re[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

Set parameter PoolSearchMode to value 1
Set parameter PoolSolutions to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
PoolSolutions  1
PoolSearchMode  1

Optimize a model with 169000 rows, 169240 columns and 433000 nonzeros
Model fingerprint: 0x30b365ff
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 100081 rows and 78272 columns
Presolve time: 0.15s
Presolved: 68919 rows, 90968 columns, 309336 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.29s

Barrier performed 0 iterations in 0.49 seconds (0.49 work units)
Barrier solve interrupted - model solved by another

In [23]:
for t, s in product(range(T), range(S)):
    if -lambda_dual[t, s] * S < P_RT[t, s] - 0.00001 or -lambda_dual[t, s] * S > P_PN[t, s] + 0.00001:
        print(f"t={t}, s={s}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

t=0, s=0, P_RT=84.12739036048507, P_IN=184.1460612244898, P_PN=182.286
t=0, s=1, P_RT=60.05393611061611, P_IN=184.1460612244898, P_PN=182.286
t=0, s=3, P_RT=69.6640903941771, P_IN=184.1460612244898, P_PN=182.286
t=0, s=4, P_RT=68.68422011146303, P_IN=184.1460612244898, P_PN=182.286
t=0, s=5, P_RT=41.064427543093444, P_IN=184.1460612244898, P_PN=182.286
t=0, s=6, P_RT=84.6575264070108, P_IN=184.1460612244898, P_PN=182.286
t=0, s=7, P_RT=29.029968688651515, P_IN=184.1460612244898, P_PN=182.286
t=0, s=8, P_RT=43.18006317947951, P_IN=184.1460612244898, P_PN=182.286
t=0, s=9, P_RT=53.60597996093772, P_IN=184.1460612244898, P_PN=182.286
t=0, s=10, P_RT=73.36670572061728, P_IN=184.1460612244898, P_PN=182.286
t=0, s=12, P_RT=78.16137132360494, P_IN=184.1460612244898, P_PN=182.286
t=0, s=13, P_RT=85.0661823713403, P_IN=184.1460612244898, P_PN=182.286
t=0, s=14, P_RT=38.068288657291454, P_IN=184.1460612244898, P_PN=182.286
t=0, s=15, P_RT=62.92703706634672, P_IN=184.1460612244898, P_PN=182.286
t

In [24]:
print(round((1/INEFF_EXT) * x_ind[:,:].sum(),2), round((1/INEFF_EXT) * x_re[:,:].sum(),2), round((1/INEFF_EXT) * x_hol[:,:].sum(),2))

29849.71 33488.23 33441.3


In [25]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_re[i, t, s] > 0.001 and dm_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, dm={dm_re[i, t, s]}")
    if zc_re[i, t, s] > 0.001 and zd_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_re[i, t, s]}, zd={zd_re[i, t, s]}")
    if dp_re[i, t, s] > 0.001 and ym_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, ym={ym_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_re[i, t, s] > 0.001 and yp_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, yp={yp_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

In [26]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print(f"\n[REPLAY] Objective Value = {OBJ_RE:.2f}, QUAD_RE = {QUAD_RE:>2f}") ; print(header)
for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT) * x_sum:>8.2f} {(1 / INEFF_EXT) * yp_avg:>8.2f} {INEFF_EXT * ym_avg:>8.2f} {INEFF_INT * dp_avg:>8.2f} {(1/INEFF_INT) * dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

# print("\n[HOLISTIC]") ; print(header)
# for t in range(T):
#     R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
#     yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
#     dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
#     zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
#     print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY] Objective Value = 5468142.02, QUAD_RE = 0.032576
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 2 |     0.00     0.00     0.00     0.00     0.00    43.30    42.87     0.00     0.00
 3 |     0.00     0.00     0.00     0.00     0.00    53.61    53.07     0.00    40.73
 4 |     0.00     0.00     0.00     0.00     0.00   102.17   101.15     0.00    91.14
 5 |     0.00     0.00     0.00     0.00     0.00     1.75     1.74     0.00   187.23
 6 |    54.55     0.00     4.25     0.00    16.77   105.95   138.26     0.00   188.88
 7 |   586.20     0.00    78.92     0.00   134.26   148.57   518.75     0.00   320.23
 8 |  1403.93   757.69    84.77     0.00   183.19   207.38   581.73     0.00

In [27]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        # 1. 내부 시장 (Internal Pool) 수급 불균형
        total_supply = np.sum(dp_re[:, t, s]) * INEFF_INT 
        total_demand = np.sum(dm_re[:, t, s]) * (1 / INEFF_INT)
        lambda_price = -lambda_dual[t, s] * S
        
        # 내부 시장 Decomposition Gap에 의한 손실
        loss = (total_demand - total_supply) * (lambda_price - P_RT[t, s])
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

# [수정 1] 외부 시장 손실 분석 (External Grid Loss) - x(DA) 포함
print("="*50) ; print("EXTERNAL GRID LOSS ANALYSIS") ; print("="*50)
total_ext_loss_energy = 0
for i, t, s in product(range(I), range(T), range(S)):
    # 1. Real-Time 판매(yp) 손실: (1/효율 - 1) * 판매량
    loss_export_rt = yp_re[i, t, s] * (1/INEFF_EXT - 1)
    
    # 2. Real-Time 구매(ym) 손실: (1 - 효율) * 구매량
    loss_import_rt = ym_re[i, t, s] * (1 - INEFF_EXT)
    
    # [NEW] Day-Ahead 판매(x) 손실 추가
    # x는 시나리오(s)와 무관하게 고정된 값이지만, 물리적 손실은 매 시나리오마다 발생함
    # x만큼 시장에 도달시키기 위해 (1/INEFF_EXT - 1)만큼 더 발전했어야 함
    loss_export_da = x_re[i, t] * (1/INEFF_EXT - 1)
    
    total_ext_loss_energy += loss_export_rt + loss_import_rt + loss_export_da

print(f"Total Physical Energy Lost in External Grid: {total_ext_loss_energy:.2f} kWh")


print(); print("="*60) ; print("SUMMARY") ; print("="*60)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Internal Aggregator Loss (Financial): {total_loss:.2f}")
print("Realized Profit (Replay + AggLoss)", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print(); print("="*60) ; print("INDIVIDUAL PROFIT ANALYSIS") ; print("="*60)
profit_ind = np.zeros(I) ; profit_re = np.zeros(I) ; profit_hol = np.zeros(I) ; profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
        lambda_price = -lambda_dual[t, s] * S
        
        # [내부 시장] 가중치 계산 (양방향 비효율 유지)
        dm_contribution = dm_re[i, t, s] * (lambda_price * (1 / INEFF_INT))
        dp_contribution = dp_re[i, t, s] * (lambda_price * INEFF_INT) 
        
        # 외부 시장 거래는 '내부 손실' 배분에 영향을 주지 않으므로 포함하지 않음
        price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    # 1. Individual Case
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t] # 정산은 도착량(x) 기준이므로 그대로 둠
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    
    # 2. Replay Case
    profit_re[i] = 0
    for t in range(T):
        # [수정 확인] Profit 계산에서는 x에 효율을 곱하지 않음 (정산은 x 그대로)
        profit_re[i] += P_DA[t] * x_re[i, t]
        
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_dual[t, :] * S
        
        # [내부 시장] 수익/비용 계산 (양방향 비효율 적용)
        profit_re[i] += np.mean([lambda_price[s] * dp_re[i, t, s] * INEFF_INT for s in range(S)]) 
        profit_re[i] -= np.mean([lambda_price[s] * dm_re[i, t, s] * (1 / INEFF_INT) for s in range(S)])
    
    # 3. Holistic Case
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_dual[t, :] * S
        
        # [내부 시장] Holistic 비교 시에도 동일하게 양방향 효율 반영
        profit_hol[i] += np.mean([lambda_price[s] * dp_hol[i, t, s] * INEFF_INT for s in range(S)])
        profit_hol[i] -= np.mean([lambda_price[s] * dm_hol[i, t, s] * (1 / INEFF_INT) for s in range(S)])

# 손실 배분 및 최종 이익 계산
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (price_weighted_usage[i] / total_price_weighted_usage)
    else:
        loss_per_player[i] = total_loss / I  
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

# 헤더 출력
print(f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Alloc Loss':<12} {'Gain ($)':<12} {'Gain (%)':<22}") 
print("-" * 125)

total_ind = 0 ; total_re = 0 ; total_hol = 0 ; total_re_adj = 0

for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    
    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_percentage_str = f"(+{percentage_change:.1f}%)" if percentage_change >= 0 else f"({percentage_change:.1f}%)"
    else:
        final_percentage_str = "(N/A)"

    print(f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f} {final_percentage_str:<22}")
    
    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_percentage_str = f"(+{total_percentage_change:.1f}%)" if total_percentage_change >= 0 else f"({total_percentage_change:.1f}%)"
else:
    total_final_percentage_str = "(N/A)"
    
print("-" * 125)
print(f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f} {total_final_percentage_str:<22}")

AGGREGATOR LOSS ANALYSIS
EXTERNAL GRID LOSS ANALYSIS
Total Physical Energy Lost in External Grid: 88379.69 kWh

SUMMARY
Individual Participation Profit 4747301.204985005
Expected Replay Profit 5468142.017962406
Internal Aggregator Loss (Financial): 126196.21
Realized Profit (Replay + AggLoss) 5594338.230829724
Holistic Profit 5468142.017962415

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 144973650.72
Player   Individual   Replay       Re+Loss      Holistic     Alloc Loss   Gain ($)     Gain (%)              
-----------------------------------------------------------------------------------------------------------------------------
0        328700.79    374765.43    383482.97    374765.43    8717.54      54782.18     (+16.7%)              
1        387826.89    436425.03    444341.11    436425.03    7916.08      56514.22     (+14.6%)              
2        593112.78    678952.29    694903.72    678952.29    15951.43     101790.94    (+17.2%)              
3        831548.26 

In [28]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        # 1. 내부 시장 (Internal Pool) 수급 불균형 (양방향 비효율 적용)
        total_supply = np.sum(dp_re[:, t, s]) * INEFF_INT 
        total_demand = np.sum(dm_re[:, t, s]) * (1 / INEFF_INT)
        
        # 내부 시장에서 걷은 돈의 단가 (Lambda)
        lambda_price = -lambda_dual[t, s] * S
        
        # 수급 불균형량 (양수: 부족해서 사야 함, 음수: 남아서 팔아야 함)
        imbalance = total_demand - total_supply
        
        # [수정 포인트] 외부 효율(INEFF_EXT)을 고려한 유효 P_RT 계산
        if imbalance > 0:
            # 부족함 -> 외부에서 사와야 함 (Buy)
            # 1만큼 부족하면 외부에서는 (1 / INEFF_EXT)만큼 사야 함 -> 비용 증가
            effective_p_rt = P_RT[t, s] * (1 / INEFF_EXT)
        else:
            # 남음 -> 외부로 팔아야 함 (Sell)
            # 1만큼 남아서 보내도 외부에는 (INEFF_EXT)만큼만 도착함 -> 수익 감소
            effective_p_rt = P_RT[t, s] * INEFF_EXT
            
        # 손실(재정적 잉여) 계산
        # (내부에서 걷은 돈) - (외부에 지불한 실질 비용)
        # loss = imbalance * lambda_price - imbalance * effective_p_rt
        loss = imbalance * (lambda_price - effective_p_rt)
        
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

# [수정 1] 외부 시장 손실 분석 (External Grid Loss) - x(DA) 포함
print("="*50) ; print("EXTERNAL GRID LOSS ANALYSIS") ; print("="*50)
total_ext_loss_energy = 0
for i, t, s in product(range(I), range(T), range(S)):
    # 1. Real-Time 판매(yp) 손실: (1/효율 - 1) * 판매량
    loss_export_rt = yp_re[i, t, s] * (1/INEFF_EXT - 1)
    
    # 2. Real-Time 구매(ym) 손실: (1 - 효율) * 구매량
    loss_import_rt = ym_re[i, t, s] * (1 - INEFF_EXT)
    
    # [NEW] Day-Ahead 판매(x) 손실 추가
    # x는 시나리오(s)와 무관하게 고정된 값이지만, 물리적 손실은 매 시나리오마다 발생함
    # x만큼 시장에 도달시키기 위해 (1/INEFF_EXT - 1)만큼 더 발전했어야 함
    loss_export_da = x_re[i, t] * (1/INEFF_EXT - 1)
    
    total_ext_loss_energy += loss_export_rt + loss_import_rt + loss_export_da

print(f"Total Physical Energy Lost in External Grid: {total_ext_loss_energy:.2f} kWh")


print(); print("="*60) ; print("SUMMARY") ; print("="*60)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Internal Aggregator Loss (Financial): {total_loss:.2f}")
print("Realized Profit (Replay + AggLoss)", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print(); print("="*60) ; print("INDIVIDUAL PROFIT ANALYSIS") ; print("="*60)
profit_ind = np.zeros(I) ; profit_re = np.zeros(I) ; profit_hol = np.zeros(I) ; profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
        lambda_price = -lambda_dual[t, s] * S
        
        # [내부 시장] 가중치 계산 (양방향 비효율 유지)
        dm_contribution = dm_re[i, t, s] * (lambda_price * (1 / INEFF_INT))
        dp_contribution = dp_re[i, t, s] * (lambda_price * INEFF_INT) 
        
        # 외부 시장 거래는 '내부 손실' 배분에 영향을 주지 않으므로 포함하지 않음
        price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    # 1. Individual Case
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t] # 정산은 도착량(x) 기준이므로 그대로 둠
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    
    # 2. Replay Case
    profit_re[i] = 0
    for t in range(T):
        # [수정 확인] Profit 계산에서는 x에 효율을 곱하지 않음 (정산은 x 그대로)
        profit_re[i] += P_DA[t] * x_re[i, t]
        
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_dual[t, :] * S
        
        # [내부 시장] 수익/비용 계산 (양방향 비효율 적용)
        profit_re[i] += np.mean([lambda_price[s] * dp_re[i, t, s] * INEFF_INT for s in range(S)]) 
        profit_re[i] -= np.mean([lambda_price[s] * dm_re[i, t, s] * (1 / INEFF_INT) for s in range(S)])
    
    # 3. Holistic Case
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_dual[t, :] * S
        
        # [내부 시장] Holistic 비교 시에도 동일하게 양방향 효율 반영
        profit_hol[i] += np.mean([lambda_price[s] * dp_hol[i, t, s] * INEFF_INT for s in range(S)])
        profit_hol[i] -= np.mean([lambda_price[s] * dm_hol[i, t, s] * (1 / INEFF_INT) for s in range(S)])

# 손실 배분 및 최종 이익 계산
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (price_weighted_usage[i] / total_price_weighted_usage)
    else:
        loss_per_player[i] = total_loss / I  
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

# 헤더 출력
print(f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Alloc Loss':<12} {'Gain ($)':<12} {'Gain (%)':<22}") 
print("-" * 125)

total_ind = 0 ; total_re = 0 ; total_hol = 0 ; total_re_adj = 0

for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    
    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_percentage_str = f"(+{percentage_change:.1f}%)" if percentage_change >= 0 else f"({percentage_change:.1f}%)"
    else:
        final_percentage_str = "(N/A)"

    print(f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f} {final_percentage_str:<22}")
    
    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_percentage_str = f"(+{total_percentage_change:.1f}%)" if total_percentage_change >= 0 else f"({total_percentage_change:.1f}%)"
else:
    total_final_percentage_str = "(N/A)"
    
print("-" * 125)
print(f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f} {total_final_percentage_str:<22}")

AGGREGATOR LOSS ANALYSIS
EXTERNAL GRID LOSS ANALYSIS
Total Physical Energy Lost in External Grid: 88379.69 kWh

SUMMARY
Individual Participation Profit 4747301.204985005
Expected Replay Profit 5468142.017962406
Internal Aggregator Loss (Financial): 123550.92
Realized Profit (Replay + AggLoss) 5591692.937567579
Holistic Profit 5468142.017962415

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 144973650.72
Player   Individual   Replay       Re+Loss      Holistic     Alloc Loss   Gain ($)     Gain (%)              
-----------------------------------------------------------------------------------------------------------------------------
0        328700.79    374765.43    383300.24    374765.43    8534.81      54599.44     (+16.6%)              
1        387826.89    436425.03    444175.18    436425.03    7750.14      56348.29     (+14.5%)              
2        593112.78    678952.29    694569.35    678952.29    15617.06     101456.57    (+17.1%)              
3        831548.26 